<a href="https://colab.research.google.com/github/Domimimi/Lab-10-NTPD/blob/main/Lab10NTPD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
import pandas as pd

In [ ]:
!pip install pyspark -q

Zadanie 1

In [ ]:
spark = SparkSession.builder \
    .appName("SparkSQL_Lab10") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
#generowanie danych
data = [("Jan", "Kowalski", 25), ("Anna", "Kowalska", 28), ("Paweł", "Nowak", 22)]
columns = ["Imie", "Nazwisko", "Wiek"]

#tworzenie tymczasowego DataFrame i zapisanie do formatu parquet
temp_df = spark.createDataFrame(data, columns)
temp_df.write.mode("overwrite").parquet("sample_data.parquet")

In [ ]:
df = spark.read.parquet("sample_data.parquet")

In [ ]:
print('Zawartość dataframe')
df.show()
print('Struktura danych')
df.printSchema()

Zawartość dataframe
+-----+--------+----+
| Imie|Nazwisko|Wiek|
+-----+--------+----+
|  Jan|Kowalski|  25|
| Anna|Kowalska|  28|
|Paweł|   Nowak|  22|
+-----+--------+----+

Struktura danych
root
 |-- Imie: string (nullable = true)
 |-- Nazwisko: string (nullable = true)
 |-- Wiek: long (nullable = true)



Zadanie 2

In [ ]:
#generowanie danych sprzedażowych
csv_content = """id,imie,miasto,wiek,kwota_zakupu
1,Jan,Warszawa,34,27.50
2,Anna,Krakow,28,12.00
3,Piotr,Gdansk,24,45.20
4,Maria,Warszawa,56,89.90
5,Krzysztof,Poznan,25,60.00"""

with open("dane_sprzedazy.csv", "w") as f:
    f.write(csv_content.strip())

In [ ]:
# wczytanie pliku CSV
df_csv = spark.read.csv(
    "dane_sprzedazy.csv",
    header=True,
    inferSchema=True
)

df_csv.printSchema()

root
 |-- id: integer (nullable = true)
 |-- imie: string (nullable = true)
 |-- miasto: string (nullable = true)
 |-- wiek: integer (nullable = true)
 |-- kwota_zakupu: double (nullable = true)



In [ ]:
#rejestracja widoku tymczasowego o nazwie
df_csv.createOrReplaceTempView("widok_sprzedazy")

In [ ]:
wynik_sql = spark.sql("SELECT * FROM widok_sprzedazy LIMIT 10")

#wyświetlenie wyniku zapytania
wynik_sql.show()

+---+---------+--------+----+------------+
| id|     imie|  miasto|wiek|kwota_zakupu|
+---+---------+--------+----+------------+
|  1|      Jan|Warszawa|  34|        27.5|
|  2|     Anna|  Krakow|  28|        12.0|
|  3|    Piotr|  Gdansk|  24|        45.2|
|  4|    Maria|Warszawa|  56|        89.9|
|  5|Krzysztof|  Poznan|  25|        60.0|
+---+---------+--------+----+------------+



Zadanie 3

In [ ]:
#generowanie danych transakcji
dane_transakcji = [
    (101, 1, "Laptop", 3500.00),
    (102, 2, "Telefon", 1500.00),
    (103, 1, "Telefon", 1750.00),
    (104, 3, "Monitor", 1200.00),
    (105, 2, " ladowarka", 80.00),
    (106, 5, "Laptop", 4000.00)
]
kolumny_transakcji = ["id_transakcji", "id_klienta", "produkt", "kwota"]

df_transakcje = spark.createDataFrame(dane_transakcji, kolumny_transakcji)
df_transakcje.createOrReplaceTempView("widok_transakcji")

#generowanie danych klienta
dane_klientow = [
    (1, "Jan Kowalski"),
    (2, "Anna Nowak"),
    (3, "Piotr Nowak"),
    (4, "Joanna Wisniewska"),
    (5, "Tomasz Pajak")
]
kolumny_klientow = ["id_klienta", "imie"]

df_klienci = spark.createDataFrame(dane_klientow, kolumny_klientow)
df_klienci.createOrReplaceTempView("widok_klientow")

In [ ]:
zapytanie = """
SELECT
    t.produkt,
    COUNT(t.id_transakcji) as liczba_transakcji,
    SUM(t.kwota) as suma_sprzedazy,
    AVG(t.kwota) as srednia_wartosc_transakcji
FROM widok_transakcji t
JOIN widok_klientow k ON t.id_klienta = k.id_klienta
WHERE t.kwota > 100
GROUP BY t.produkt
"""

wynik = spark.sql(zapytanie)
wynik.show()

+-------+-----------------+--------------+--------------------------+
|produkt|liczba_transakcji|suma_sprzedazy|srednia_wartosc_transakcji|
+-------+-----------------+--------------+--------------------------+
| Laptop|                2|        7500.0|                    3750.0|
|Telefon|                2|        3250.0|                    1625.0|
|Monitor|                1|        1200.0|                    1200.0|
+-------+-----------------+--------------+--------------------------+



In [ ]:
wynik.write.mode("overwrite").parquet("zad3.parquet")